# Project 1 - Explanatory Visualization
### David González & Rebeca Torrecilla
#### Information Visualization 2025-2026

---



# Preparing and cleaning data

## Loading and merging

In [1]:
import pandas as pd
import io
import altair as alt
from vega_datasets import data
import re
from google.colab import files

# We import the NSF terminations table, the list of flagged words and Ted Cruz's list
uploaded = files.upload()

Saving cruz_list.csv to cruz_list.csv
Saving flagged_words_trump_admin.csv to flagged_words_trump_admin.csv
Saving nsf_terminations_airtable.csv to nsf_terminations_airtable.csv


In [2]:
# Conversion to DataFrames
df_nsf_terminations = pd.read_csv(io.BytesIO(uploaded['nsf_terminations_airtable.csv']))
df_cruz_list = pd.read_csv(io.BytesIO(uploaded['cruz_list.csv']))

In [3]:
# Join of the NSF terminated grants with the Ted's Cruz list
df_cruz_list[['grant_number', 'in_cruz_list']] = df_cruz_list['grant_number;in_cruz_list'].str.split(';', expand=True)
df_cruz_list['grant_number'] = pd.to_numeric(df_cruz_list['grant_number'])

df_merged = pd.merge(df_nsf_terminations, df_cruz_list, left_on='grant_id', right_on='grant_number', how='left')

# We put 'False' instead of NaNs
df_merged['in_cruz_list'] = df_merged['in_cruz_list'].fillna(False)

df_merged['in_cruz_list'] = (df_merged['in_cruz_list'] == 'TRUE') | (df_merged['in_cruz_list'] == True)

print(df_merged['in_cruz_list'].value_counts())

in_cruz_list
False    1503
True      467
Name: count, dtype: int64


In [4]:
# Elimination of duplicated columns
df_merged = df_merged.drop(columns=['grant_number', 'grant_number;in_cruz_list'])

## Exploratory analysis and cleaning

In [5]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1970 entries, 0 to 1969
Data columns (total 36 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   grant_id                 1970 non-null   int64  
 1   status                   1970 non-null   object 
 2   terminated               1970 non-null   bool   
 3   suspended                1970 non-null   bool   
 4   termination_date         1970 non-null   object 
 5   reinstated               1970 non-null   bool   
 6   reinstatement_date       420 non-null    object 
 7   reinstatement_indicator  416 non-null    object 
 8   nsf_url                  1970 non-null   object 
 9   usaspending_url          1970 non-null   object 
 10  project_title            1970 non-null   object 
 11  abstract                 1969 non-null   object 
 12  org_name                 1970 non-null   object 
 13  org_state                1970 non-null   object 
 14  org_city                

In [6]:
# We inspect if there are duplicated rows
print(f'There are {df_merged.duplicated(subset=['grant_id']).sum()} duplicated rows.')

# Count the null values
print()
print('For each column we have the next null values:')
print(df_merged.isnull().sum())


There are 0 duplicated rows.

For each column we have the next null values:
grant_id                      0
status                        0
terminated                    0
suspended                     0
termination_date              0
reinstated                    0
reinstatement_date         1550
reinstatement_indicator    1554
nsf_url                       0
usaspending_url               0
project_title                 0
abstract                      1
org_name                      0
org_state                     0
org_city                      1
award_type                    0
usa_start_date              711
usa_end_date                711
nsf_start_date                0
nsf_end_date                  0
nsf_program_name              1
nsf_primary_program           0
usa_nsf_office              711
nsf_total_budget              0
nsf_obligated                 0
usaspending_obligated       711
usaspending_outlaid         711
estimated_budget              0
estimated_outlays           

We do not worry about this null data as all those coluns that show a high number of NaNs will not be used for the analysis.

In [7]:
# Data conversion
dates = [df_merged['nsf_start_date'],
         df_merged['nsf_end_date'],
         df_merged['termination_date'],
         df_merged['reinstatement_date'],
         df_merged['usa_start_date'],
         df_merged['usa_end_date']
         ]

for data in dates:
  data = pd.to_datetime(data, errors='coerce')

# Columns with limited unique values can be more efficient as 'category'
df_merged['in_cruz_list'] = df_merged['in_cruz_list'].astype('category')

In [8]:
# Elimination of innecessary columns
df_merged = df_merged.drop(columns=['suspended', 'termination_date', 'reinstatement_date',
                                    'reinstatement_indicator', 'nsf_url', 'usaspending_url',
                                    'award_type', 'usa_start_date',
                                    'usa_end_date', 'nsf_program_name', 'nsf_primary_program',
                                    'usa_nsf_office', 'nsf_obligated', 'usaspending_obligated',
                                    'usaspending_outlaid', 'estimated_budget', 'estimated_outlays',
                                    'estimated_remaining', 'division', 'directorate', 'div',
                                    'dir', 'record_sha1'])

In [ ]:
# Save cleaned data
df_merged.to_csv('df_clean.csv', index=False)
files.download('df_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Questions

## Q1: How are the cancellations distributed by states?

### Choropletic Map

In [9]:
# Cancellations by state
cancellations_by_state = df_merged['org_state'].value_counts().reset_index()
cancellations_by_state.columns = ['State', 'Cancellations']
cancellations_by_state

,State,Cancellations
0,CA,466
1,MA,256
2,TX,122
3,NY,102
4,FL,59
5,IL,58
6,PA,55
7,VA,55
8,NC,55
9,GA,54


In [10]:
total_cancellations = cancellations_by_state['Cancellations'].sum()

top_15_cancellations = cancellations_by_state.sort_values(by='Cancellations', ascending=False).head(15).copy()

top_15_cancellations['Percentage of Total'] = (top_15_cancellations['Cancellations'] / total_cancellations) * 100

top_15_cancellations['Percentage Label'] = top_15_cancellations['Percentage of Total'].apply(lambda x: f'{x:.1f}%')

print("Top 15 States with Most Cancelled Grants:")
print(top_15_cancellations)

Top 15 States with Most Cancelled Grants:
   State  Cancellations  Percentage of Total Percentage Label
0     CA            466            23.654822            23.7%
1     MA            256            12.994924            13.0%
2     TX            122             6.192893             6.2%
3     NY            102             5.177665             5.2%
4     FL             59             2.994924             3.0%
5     IL             58             2.944162             2.9%
6     PA             55             2.791878             2.8%
7     VA             55             2.791878             2.8%
8     NC             55             2.791878             2.8%
9     GA             54             2.741117             2.7%
10    DC             53             2.690355             2.7%
11    MI             53             2.690355             2.7%
12    CO             52             2.639594             2.6%
13    MD             48             2.436548             2.4%
14    WA             45     

In [11]:
from vega_datasets import data

# We add the Wyoming state as an state without grant cancellations
new_row = pd.DataFrame([{'State': 'WY', 'Cancellations': 0}])
cancellations_by_state = pd.concat([cancellations_by_state, new_row], ignore_index=True)

# We eliminate Puerto Rico and Virgin Islands states
territories_to_exclude = ['PR', 'VI']
cancellations_filtered = cancellations_by_state[~cancellations_by_state['State'].isin(territories_to_exclude)].copy()

# Conversion to the numeric representation verga_datasets uses for the states
state_to_fips = {
    'AL': 1, 'AK': 2, 'AZ': 4, 'AR': 5, 'CA': 6, 'CO': 8, 'CT': 9, 'DE': 10, 'DC': 11,
    'FL': 12, 'GA': 13, 'HI': 15, 'ID': 16, 'IL': 17, 'IN': 18, 'IA': 19, 'KS': 20,
    'KY': 21, 'LA': 22, 'ME': 23, 'MD': 24, 'MA': 25, 'MI': 26, 'MN': 27, 'MS': 28,
    'MO': 29, 'MT': 30, 'NE': 31, 'NV': 32, 'NH': 33, 'NJ': 34, 'NM': 35, 'NY': 36,
    'NC': 37, 'ND': 38, 'OH': 39, 'OK': 40, 'OR': 41, 'PA': 42, 'RI': 44, 'SC': 45,
    'SD': 46, 'TN': 47, 'TX': 48, 'UT': 49, 'VT': 50, 'VA': 51, 'WA': 53, 'WV': 54,
    'WI': 55, 'WY': 56
}

# We add the IDs column to our dataset
cancellations_filtered['id'] = cancellations_filtered['State'].map(state_to_fips)

color_scale=alt.Scale(scheme='reds')

# Complete map
states_map = alt.topo_feature(data.us_10m.url, 'states')

choropletic_map = alt.Chart(states_map).mark_geoshape(
    stroke='white', strokeWidth=0.5
).project(
    type='albersUsa'
).encode(
    color=alt.Color('Cancellations:Q', title='Grant rejections', scale=color_scale, legend=alt.Legend(orient="left")),
    tooltip=[
        alt.Tooltip('State:N', title='State'),
        alt.Tooltip('Cancellations:Q', title='Cancellations', format=',')
    ]
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(cancellations_filtered, 'id', ['Cancellations', 'State'])
).properties(
    width=700,
    height=400,
    title='Distribution of grant cancellations by state'
)

In [12]:
choropletic_map

alt.Chart(...)

### Bar Chart

In [13]:
bch_1 = alt.Chart(top_15_cancellations).mark_bar(color='#bf3d2a').encode(
    x=alt.X('Cancellations:Q', title='Number of Cancellations'),
    y=alt.Y('State:N',
            title='State',
            sort='-x'
           ),
    tooltip=[
        alt.Tooltip('State:N', title='State'),
        alt.Tooltip('Cancellations:Q', title='Cancellations', format=','),
        alt.Tooltip('Percentage Label:N', title='% of Total Cancellations')
    ]
).properties(
    title='Top 15 with the Most Cancelled Grants',
    width=600,
    height=400
)

bch_1

alt.Chart(...)

### Bubble map

In [14]:
capitals_df = data.us_state_capitals()

state_name_to_abbr = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA',
    'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH',
    'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY', 'North Carolina': 'NC',
    'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA',
    'Rhode Island': 'RI', 'South Carolina': 'SC', 'South Dakota': 'SD', 'Tennessee': 'TN',
    'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA',
    'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY',
}

capitals_df['State'] = capitals_df['state'].map(state_name_to_abbr)

capitals_df = capitals_df[['State', 'lat', 'lon']]
capitals_df.columns = ['State', 'latitude', 'longitude']

data_for_circles = pd.merge(cancellations_filtered, capitals_df, on='State')

print(data_for_circles.head())

  State  Cancellations  id   latitude   longitude
0    CA            466   6  38.555605 -121.468926
1    MA            256  25  42.235200  -71.027500
2    TX            122  48  30.266667  -97.750000
3    NY            102  36  42.659829  -73.781339
4    FL             59  12  30.451800  -84.272770


In [15]:
# Base map
base_map = alt.Chart(states_map).mark_geoshape(
    fill='lightgray',
    stroke='white'
).project(
    type='albersUsa'
).properties(
    width=700,
    height=400
)

# Circle chart
circles = alt.Chart(data_for_circles).mark_circle(
    opacity=0.6,
    stroke='black',
    strokeWidth=0.5
).encode(
    longitude='longitude:Q',
    latitude='latitude:Q',

    size=alt.Size('Cancellations:Q',
                  title='Number of Cancellations',
                  scale=alt.Scale(range=[10, 5000]),
                  legend=alt.Legend(orient="left")
                 ),

    color=alt.Color('Cancellations:Q',
                    scale=alt.Scale(scheme='reds')
                   ),

    tooltip=[
        alt.Tooltip('State:N', title='Estado'),
        alt.Tooltip('Cancellations:Q', title='Cancellations', format=',')
    ]
)

bubble_map = (base_map + circles).properties(
    title='Distribution of Grant Cancellations by State (Proportional Symbols)'
)

In [ ]:
bubble_map

alt.LayerChart(...)

I like that the circles are centered in the capital's locations but is not easy to compare magnitudes.

### Choropletic map based on the political parties

In [16]:
# We create a dictionary with the association of each state with their currrent political party
party_map = {
    'AL': 'Republican', 'AK': 'Republican', 'AZ': 'Democrat', 'AR': 'Republican', 'CA': 'Democrat',
    'CO': 'Democrat', 'CT': 'Democrat', 'DE': 'Democrat', 'DC': 'Democrat', 'FL': 'Republican',
    'GA': 'Democrat', 'HI': 'Democrat', 'ID': 'Republican', 'IL': 'Democrat', 'IN': 'Republican',
    'IA': 'Republican', 'KS': 'Republican', 'KY': 'Republican', 'LA': 'Republican', 'ME': 'Democrat',
    'MD': 'Democrat', 'MA': 'Democrat', 'MI': 'Democrat', 'MN': 'Democrat', 'MS': 'Republican',
    'MO': 'Republican', 'MT': 'Republican', 'NE': 'Republican', 'NV': 'Democrat', 'NH': 'Democrat',
    'NJ': 'Democrat', 'NM': 'Democrat', 'NY': 'Democrat', 'NC': 'Republican', 'ND': 'Republican',
    'OH': 'Republican', 'OK': 'Republican', 'OR': 'Democrat', 'PA': 'Democrat', 'RI': 'Democrat',
    'SC': 'Republican', 'SD': 'Republican', 'TN': 'Republican', 'TX': 'Republican', 'UT': 'Republican',
    'VT': 'Democrat', 'VA': 'Democrat', 'WA': 'Democrat', 'WV': 'Republican', 'WI': 'Democrat', 'WY': 'Republican'
}

cancellations_filtered['Party'] = cancellations_filtered['State'].map(party_map)
print(cancellations_filtered.head())

  State  Cancellations  id       Party
0    CA            466   6    Democrat
1    MA            256  25    Democrat
2    TX            122  48  Republican
3    NY            102  36    Democrat
4    FL             59  12  Republican


In [17]:
# Background base
base_map = alt.Chart(states_map).mark_geoshape(
    fill='lightgray',
    stroke='white'
).properties(
    width=700,
    height=400
)

In [18]:
# We create a domain range for the graphics' legend
max_cancellations = cancellations_filtered['Cancellations'].max()
domain_range = [0, max_cancellations]

In [19]:
# Republicans map, we only take into account those states that are currectly republican
red_states_layer = alt.Chart(states_map).mark_geoshape(
    stroke='white'
).encode(
    color=alt.Color('Cancellations:Q',
                    scale=alt.Scale(scheme='reds', domain=domain_range),
                    legend=alt.Legend(title=['Cancellations', '(Republicans)'], orient='left' )
                   ),
    tooltip=[
        alt.Tooltip('State:N'), alt.Tooltip('Cancellations:Q', format=','), alt.Tooltip('Party:N')
    ]
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(cancellations_filtered, 'id', ['Cancellations', 'State', 'Party'])
).transform_filter(
    alt.datum.Party == 'Republican'
)

In [20]:
# Democrats map
blue_states_layer = alt.Chart(states_map).mark_geoshape(
    stroke='white'
).encode(
    color=alt.Color('Cancellations:Q',
                    scale=alt.Scale(scheme='blues', domain=domain_range),
                    legend=alt.Legend(title=['Cancellations', '(Democrats)'], orient='left')
                   ),
    tooltip=[
        alt.Tooltip('State:N'), alt.Tooltip('Cancellations:Q', format=','), alt.Tooltip('Party:N')
    ]
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(cancellations_filtered, 'id', ['Cancellations', 'State', 'Party'])
).transform_filter(
    alt.datum.Party == 'Democrat'
)

In [21]:
dichromatic_map = (base_map + red_states_layer + blue_states_layer).project(
    type='albersUsa'
).properties(
    title='Cancelled grants distribution based on the state and the politician affiliation',
    width=800,
    height=500
).resolve_scale(
    color='independent'
)

dichromatic_map

alt.LayerChart(...)

### Bar chart based on the political affiliation

In [23]:
top_15_cancellations_filtered = cancellations_filtered.sort_values(by='Cancellations', ascending=False).head(15).copy()

bar_chart_political = alt.Chart(top_15_cancellations_filtered).mark_bar().encode(
    x=alt.X('Cancellations:Q', title='Number of Cancellations'),
    y=alt.Y('State:N',
            title='State',
            sort='-x'
           ),

    color=alt.Color('Party:N',
                    title='Political Party',
                    scale=alt.Scale(domain=['Republican', 'Democrat'],
                                    range=['#bf3d2a', '#4c78a8']),
                    legend = None
                   ),

    tooltip=[
        alt.Tooltip('State:N'),
        alt.Tooltip('Party:N'),
        alt.Tooltip('Cancellations:Q', format=','),
        alt.Tooltip('Percentage Label:N', title='Total % of Cancellations')
    ]
).properties(
    title='The 15 states with the most cancellations (grouped by political party).',
    width=500,
    height=500
)

bar_chart_political

alt.Chart(...)

### Lineal chart

In [24]:
time_series_data = df_merged[['nsf_start_date', 'nsf_end_date', 'org_state']].copy()

# Conversion to datetime type
time_series_data['start'] = pd.to_datetime(time_series_data['nsf_start_date'], errors='coerce')
time_series_data['end'] = pd.to_datetime(time_series_data['nsf_end_date'], errors='coerce')

time_series_data['Party'] = time_series_data['org_state'].map(party_map)

# Drop NaNs
time_series_data = time_series_data.dropna(subset=['start', 'end', 'Party'])

print(time_series_data.head())

  nsf_start_date nsf_end_date org_state      start        end       Party
0     2022-07-01   2025-04-18        PA 2022-07-01 2025-04-18    Democrat
1     2023-09-01   2025-04-18        TX 2023-09-01 2025-04-18  Republican
2     2023-04-01   2025-04-18        NM 2023-04-01 2025-04-18    Democrat
3     2023-07-01   2028-06-30        CA 2023-07-01 2028-06-30    Democrat
4     2022-08-01   2025-04-18        OK 2022-08-01 2025-04-18  Republican


In [25]:
starts = time_series_data[['start', 'Party']].copy()
starts = starts.rename(columns={'start': 'Date'})
starts['Change'] = 1

ends = time_series_data[['end', 'Party']].copy()
ends = ends.rename(columns={'end': 'Date'})
ends['Change'] = -1

events_df = pd.concat([starts, ends], ignore_index=True)

events_df = events_df.sort_values('Date')

In [26]:
events_df['Active_Grants'] = events_df.groupby('Party')['Change'].cumsum()

# Final table
print(events_df.head())

          Date       Party  Change  Active_Grants
579 2013-10-01    Democrat       1              1
250 2014-10-01  Republican       1              1
274 2016-08-15    Democrat       1              2
580 2017-06-01  Republican       1              2
583 2017-07-01    Democrat       1              3


In [27]:
line_chart = alt.Chart(events_df).mark_line().encode(
    x=alt.X('Date:T'),
    y=alt.Y('Active_Grants:Q', title='Number of active grants'),

    color=alt.Color('Party:N',
                    title='State\'s party',
                    scale=alt.Scale(domain=['Republican', 'Democrat'],
                                    range=['#bf3d2a', '#4c78a8'])
                   ),

    tooltip=[
        alt.Tooltip('Date:T'),
        alt.Tooltip('Party:N'),
        alt.Tooltip('Active_Grants:Q', title='Active grants')
    ]
).properties(
    title='Trend in the Number of Grants Cancelled by the NSF over Time',
    width=800,
    height=400
).interactive()

line_chart

alt.Chart(...)

It seems like there are a lot more grants in the democrats than in the republicans side but it's important to remember that this chart doesn't represent the total grants of the USA but only those that ended up being cancelled (or later reinstated) by the NSF.

### Final visualization

In [28]:
dichromatic_map

alt.LayerChart(...)

#### Justification
To answer Q1, a **dichromatic coropleth map** was designed. This chart type was chosen for its efficiency in encoding two distinct variables (one categorical and one quantitative) within a single geographic view, thus achieving a **high data-ink ratio**.

This design leverages two visual channels: color hue represents the state's political affiliation (red for Republican and blue for Democrat), while color intensity encodes the number of grant cancellations.

Alternative chart types were considered. Bar charts, while precise for comparison, would completely eliminate the crucial geographic context. Bubble maps preserve geographically but introduce challenges with occlusion, and comparing circle areas is inherently less accurate for the user than perceiving variations in color intensity.

With this final map, a user can answer the question intuitively: the darkest states are those most affected. Furthermore, the color hue enables deeper analysis by revealing political patterns. For instance, the map immediately highlights that the impact is severe in both Democratic states, California and Massachusetts, suggesting the issue **may be related to the political affiliations**.

##Q2: What are the institutions that have been more affected in terms of number of cancelled grants? How does this compare to the others?

### Bar chart

#### Most Affected Institutions versus All the Others

In [29]:
# We count the cancellations for each institution
cancellations_by_institution = df_merged['org_name'].value_counts()

cancellations_by_institution = cancellations_by_institution.reset_index()
cancellations_by_institution.columns = ['Institution', 'Number of Cancellations']

print(cancellations_by_institution.head(10))

# With this we can see the total number of rows (all the diferents institutions we dispose)
print("")
print(f"We have a total of {cancellations_by_institution.shape[0]} different institutions.")

                                         Institution  Number of Cancellations
0               University of California-Los Angeles                      306
1                                 Harvard University                      199
2  Regents of the University of Michigan - Ann Arbor                       28
3                           Arizona State University                       27
4                  University of Colorado at Boulder                       24
5                          Michigan State University                       17
6                    University of Wisconsin-Madison                       16
7                   Florida International University                       16
8                           University of Washington                       16
9                    University of California-Irvine                       15

We have a total of 507 different institutions.


In [30]:
# Data (only) of the insituttions with the most cancelled grants
top_n = 15
top_n_institutions_bc = cancellations_by_institution.head(15).copy()

print(top_n_institutions_bc)

                                          Institution  Number of Cancellations
0                University of California-Los Angeles                      306
1                                  Harvard University                      199
2   Regents of the University of Michigan - Ann Arbor                       28
3                            Arizona State University                       27
4                   University of Colorado at Boulder                       24
5                           Michigan State University                       17
6                     University of Wisconsin-Madison                       16
7                    Florida International University                       16
8                            University of Washington                       16
9                     University of California-Irvine                       15
10                        University of South Florida                       15
11                      University of Texas at Austi

In [31]:
# Data of the institutions with the most cancelled grants compared to the rest
top_institutions = cancellations_by_institution.head(top_n)

others_sum = cancellations_by_institution.iloc[top_n:]['Number of Cancellations'].sum()
others_row = pd.DataFrame([{'Institution': 'All Others', 'Number of Cancellations': others_sum}])

cancellations_comp = pd.concat([top_institutions, others_row], ignore_index=True)

print(cancellations_comp)

                                          Institution  Number of Cancellations
0                University of California-Los Angeles                      306
1                                  Harvard University                      199
2   Regents of the University of Michigan - Ann Arbor                       28
3                            Arizona State University                       27
4                   University of Colorado at Boulder                       24
5                           Michigan State University                       17
6                     University of Wisconsin-Madison                       16
7                    Florida International University                       16
8                            University of Washington                       16
9                     University of California-Irvine                       15
10                        University of South Florida                       15
11                      University of Texas at Austi

In [32]:
bar_chart_2 = alt.Chart(cancellations_comp).mark_bar().encode(
    x=alt.X('Number of Cancellations:Q', title='Number of Cancellations'),
    y=alt.Y('Institution:N', title='Institution',
            sort='-x'
           ),
    color = alt.condition(
        alt.datum.Institution == 'All Others',
        alt.value('lightgray'),
        alt.value('#bf3d2a')
    ),

    tooltip=[
        alt.Tooltip('Institution:N', title='Institution'),
        alt.Tooltip('Number of Cancellations:Q', title='Cancellations', format=',')
    ]
).properties(
    title='Most Affected Institutions due to NSF Grant Cancellations vs All the others',
    width=600,
    height=400
)

bar_chart_2

alt.Chart(...)

Doesn't really convences me the fact that the some red bars seem very small in this view.

####Most affected institutions with a general average dotted line

In [33]:
bar_chart_2 = alt.Chart(top_n_institutions_bc).mark_bar().encode(
    x=alt.X('Number of Cancellations:Q', title='Number of Cancellations'),
    y=alt.Y('Institution:N', title='Institution',
            sort='-x'
           ),
    color = alt.value('#bf3d2a'),

    tooltip=[
        alt.Tooltip('Institution:N', title='Institution'),
        alt.Tooltip('Number of Cancellations:Q', title='Cancellations', format=',')
    ]
).properties(
    title='Most Affected Institutions due to NSF Grant Cancellations',
    width=600,
    height=400
)

# We add a dotted line that marks the average number of cancellations
average_line = alt.Chart(cancellations_by_institution).mark_rule(
    color='lightgrey',
    strokeDash=[5, 3], # Dotted line
    size=2
).encode(
    x='average(Number of Cancellations):Q'
)

layered_bar_chart = (bar_chart_2 + average_line)

layered_bar_chart

alt.LayerChart(...)

This chart seems better that the last one as the metrics of the most affected institutions are more visible now. We can still add more information to it.

#### Most affected institutions with an average line and political affiliation

In [34]:
institution_to_state = df_merged[['org_name', 'org_state']].drop_duplicates()
institution_to_state.columns = ['Institution', 'State']

cancellations_with_state = pd.merge(cancellations_by_institution, institution_to_state, on='Institution')

cancellations_with_state['Party'] = cancellations_with_state['State'].map(party_map)

top_cancellations_with_party = cancellations_with_state.head(15).copy()

In [35]:
# Bar chart with color hue based on political affiliations
bar_chart_2_political = alt.Chart(top_cancellations_with_party).mark_bar().encode(
    x=alt.X('Number of Cancellations:Q', title='Number of Cancellations'),
    y=alt.Y('Institution:N', title='Institution',
            sort='-x'
           ),

    color=alt.Color('Party:N',
                    title='Political Party',
                    scale=alt.Scale(domain=['Republican', 'Democrat'],
                                    range=['#bf3d2a', '#4c78a8']),
                    legend=alt.Legend(orient="right")
                   ),

    tooltip=[
        alt.Tooltip('Institution:N'),
        alt.Tooltip('Party:N', title='Party'),
        alt.Tooltip('Number of Cancellations:Q', title='Cancellations', format=',')
    ]
).properties(
    title=f'Top {top_n} Most Affected Institutions by Grant Cancellations',
    width=600,
    height=400
)

In [36]:
# We create and average line to serve as a benchmark
average_grants = cancellations_by_institution['Number of Cancellations'].mean()
average_text = f"{average_grants:.1f}"

# We also put the specific value next to the legend for further inspection
label_av = alt.Chart().mark_text(
    text='Average:',
    align='right',
    fontWeight='bold',
    fontSize=11,
    x=665,
    y=60
)

value_av = alt.Chart().mark_text(
    text=average_text,
    align='left',
    fontSize=11,
    x=670,
    y=60
)

average_line = alt.Chart(cancellations_by_institution).mark_rule(
    color='black',
    strokeDash=[5, 3],
    size=2
).encode(
    x='average(Number of Cancellations):Q'
)

In [37]:
layered_bar_chart_political = (bar_chart_2_political + average_line + value_av + label_av)

layered_bar_chart_political

alt.LayerChart(...)

#### Cumulative bar chart

In [38]:
data_for_bar_chart = cancellations_by_institution.head(top_n).copy()
sum_top_n = data_for_bar_chart['Number of Cancellations'].sum()
sum_others = cancellations_by_institution.iloc[top_n:]['Number of Cancellations'].sum()

data_15vsall = pd.DataFrame({
    'Group': [f'Top {top_n} Institutions', 'All the others'],
    'Value': [sum_top_n, sum_others]
})

bar_15vsall = alt.Chart(data_15vsall).mark_bar().encode(
    y=alt.Y('Value:Q', title='Number of Cancellations'),
    x=alt.X('Group:N', title='Group',
            sort='-y',
            axis=alt.Axis(labelAngle=0)
           ),
    color = alt.Color('Group:N',
                    title='Group',
                    scale=alt.Scale(domain=[f'Top {top_n} Institutions', 'All the others'],
                                    range=['#bf3d2a', '#d3d3d3']),
                    legend=alt.Legend(orient="right")
                    ),
    tooltip=[
        'Group:N',
        'Value:Q'
    ]
).properties(
    width=150,
    height=400
)

bar_15vsall

alt.Chart(...)

This bar chart shows perfectly the relation between the most affected institutions and all the others. It's surprising how much cancelled grants are included only in 15 institutions, almost 500 grants away from the 'All the others' category, that contains 492 institutions.

### Histogram

In [39]:
basic_histogram = alt.Chart(cancellations_by_institution).mark_bar().encode(
    x=alt.X('Number of Cancellations:Q',
            bin=alt.Bin(maxbins=50),
           ),
    y=alt.Y('count()', title='Number of Institutions'),
    color=alt.value('#bf3d2a'),

    tooltip=[
        alt.Tooltip('count()', title='Number of Institutions'),
        alt.Tooltip('Number of Cancellations:Q', bin=True, title='Cancellations range')
    ]
).properties(
    title='Distribution of the Cancellations by Institution',
    width=400
)

basic_histogram

alt.Chart(...)

Great for visualizing distributions but the outliers are almost impossible to see.

In [40]:
majority_chart = alt.Chart(cancellations_by_institution).mark_bar().encode(
    x=alt.X('Number of Cancellations:Q', bin=alt.Bin(maxbins=30), title=None),
    y=alt.Y('count()', title='Number of Institutions', scale=alt.Scale(domain=[0, 300])),
    color=alt.value('#bf3d2a'),
    tooltip=[
        alt.Tooltip('count()', title='Number of Institutions'),
        alt.Tooltip('Number of Cancellations:Q', bin=True, title='Cancellations range')]
).transform_filter(
    alt.datum['Number of Cancellations'] < 30  # Filtering
).properties(
    title='Distribution of Most Part of the Institutions (< 30 Cancellations)',
    width=350
)

# Outliers' histogram
outliers_chart = alt.Chart(cancellations_by_institution).mark_bar().encode(
    x=alt.X('Number of Cancellations:Q',  bin=alt.Bin(maxbins=12), scale=alt.Scale(domain=[170, 350])),
    y=alt.Y('count()', title=None, scale=alt.Scale(domain=[0, 1.2])),
    color=alt.value('#bf3d2a'),
    tooltip=[
        alt.Tooltip('count()', title='Number of Institutions'),
        alt.Tooltip('Number of Cancellations:Q', bin=True, title='Cancellations range')]
).transform_filter(
    alt.datum['Number of Cancellations'] >= 30
).properties(
    title='Outliers (≥ 30 Cancellations)',
    width=300
)

combined_view = majority_chart | outliers_chart

combined_view

alt.HConcatChart(...)

This way we can visualize the outliers though I'm afraid some people may confuse the values of the Y axis in the second graph (it can seem that there are a lot of institutions that have a high number of cancelled grants).

### Dot plot

In [41]:
top_30_institutions = cancellations_by_institution.head(30)

dot_plot = alt.Chart(top_30_institutions).mark_point(
    filled=True,
    size=80
).encode(
    x=alt.X('Number of Cancellations:Q'),
    y=alt.Y('Institution:N',
            sort='-x'
           ),
    color=alt.value('#bf3d2a'),
    tooltip=[
        alt.Tooltip('Institution:N'),
        alt.Tooltip('Number of Cancellations:Q', format=',')
    ]
).properties(
    title='Top 30 Most Affected Institutions in Terms of Grant Cancellations (Dot Plot)',
    width=600,
    height=500
)

dot_plot

alt.Chart(...)

Doesn't differ too much from the bar chart, furthermore the points are more difficult to perceive than the bars.

### Strip plot

In [42]:
strip_plot = alt.Chart(cancellations_by_institution).mark_tick(
    thickness=5,
    size=80
).encode(
    x=alt.X('Number of Cancellations:Q'),
    color=alt.value('#bf3d2a'),
    tooltip=[
        alt.Tooltip('Institution:N'),
        alt.Tooltip('Number of Cancellations:Q')
    ]
).properties(
    title='Distribution of Cancellation by Institution (Strip Plot)',
    width=600,
    height=100
)

strip_plot

alt.Chart(...)

Although the view is interesting and permits to visualize outliers, it has a great problem with overlap.

### Jitter plot

In [43]:
jitter_plot = alt.Chart(cancellations_by_institution).mark_circle(
    opacity=0.6,
    size=80
).encode(
    x=alt.X('Number of Cancellations:Q'),

    y=alt.Y('jitter:Q',
            title=None,
            axis=alt.Axis(values=[0], ticks=False, grid=False, labels=False)
           ),
    color=alt.value('#bf3d2a'),
    tooltip=[
        alt.Tooltip('Institution:N'),
        alt.Tooltip('Number of Cancellations:Q')
    ]
).transform_calculate(
    jitter='random()'
).properties(
    title='Distribution of Cancellations by Institution (Jitter Plot)',
    width=850,
    height=150
)

average_line_strip = alt.Chart(cancellations_by_institution).mark_rule(
    color='lightgrey',
    strokeDash=[5, 3],
    size=2
).encode(
    x='average(Number of Cancellations):Q'
)

layered_strip_plot = (jitter_plot + average_line_strip)

layered_strip_plot

alt.LayerChart(...)

The jitter plot shows a little bit better the amount of institutions with low numbers of cancelled grants but still, wouldn't be a good option to use only this chart for the visualization as it still contains a lot of overlap.

### Final Visualization

In [44]:
layered_bar_chart_political

alt.LayerChart(...)

#### Justification
To answer Q2, a **horizontal bar chart** was selected as the optimal visualization for ranking and comparing the most affected institutions. This format is ideal for the task as it provides ample space for long institutional names, **ensuring legibility** without rotation or truncation. The length of each bar directly encodes the number of cancellations, making magnitude comparisons straightforward.

The chart is enhanced with two additional layers of information to **provide deeper context**. First, color hue is used to encode the political affiliation of each institution's state (red for Republican and blue for Democrat), adding a crucial analytical dimension. Second, a dotted vertical line represents the average number of cancellations across all 507 institutions. This line serves as a **critical contextual benchmark**, allowing a user to immediately gauge how extreme these top outliers are relative to the overall norm. This average value is explicitly annotated for precise reference.

This multi-layer design enables a user to answer the question comprehensively. They can identify their standing relative to the average. For example, a user can quickly observe that the vast majority of the top-ranked institutions are located in Democratic states.



## Q3: What are the institutions that have been more affected in terms of budget and how does this compare to the others?

### Bar chart

#### Most affected institutions versus the rest

In [45]:
budget_by_institution = df_merged.groupby('org_name')['nsf_total_budget'].sum()
budget_by_institution = budget_by_institution.sort_values(ascending=False).reset_index()

budget_by_institution.columns = ['Institution', 'Total Budget Cancelled']

print(budget_by_institution.head(15))

                                          Institution  Total Budget Cancelled
0                University of California-Los Angeles               199678586
1                                  Harvard University               149391352
2                            Arizona State University                36774883
3                   University of Colorado at Boulder                27347170
4                   University of California-Berkeley                23252270
5                      University of Texas at El Paso                21304493
6                       University of Texas at Austin                20581798
7   Regents of the University of Michigan - Ann Arbor                17569315
8                     University of California-Irvine                17553149
9                            University of Washington                16739533
10                                  Auburn University                16430719
11            University of Maryland Baltimore County           

In [46]:
top_n = 15

# Bar Chart data
top_budget = budget_by_institution.head(top_n).copy()
top_budget['Group'] = f'Top {top_n} Institutions'

In [47]:
sum_top_n = top_budget['Total Budget Cancelled'].sum()
sum_others = budget_by_institution.iloc[top_n:]['Total Budget Cancelled'].sum()

grouped_budget = pd.DataFrame({
    'Group': [f'Top {top_n} Institutions', 'All the others'],
    'Value': [sum_top_n, sum_others]
})

In [48]:
# Percentage calculation
total_value = grouped_budget['Value'].sum()

grouped_budget['Percentage'] = (grouped_budget['Value'] / total_value) * 100
grouped_budget['TextLabel'] = grouped_budget['Percentage'].apply(lambda x: f'{x:.1f}%')

print(grouped_budget)

                 Group       Value  Percentage TextLabel
0  Top 15 Institutions   607594637   35.385401     35.4%
1       All the others  1109482538   64.614599     64.6%


In [49]:
others_sum_budget = budget_by_institution.iloc[top_n:]['Total Budget Cancelled'].sum()

others_row_budget = pd.DataFrame([{'Institution': 'All Others', 'Total Budget Cancelled': others_sum_budget}])

budget_vs_others = pd.concat([top_institutions, others_row], ignore_index=True)

print(budget_vs_others)

                                          Institution  Number of Cancellations
0                University of California-Los Angeles                      306
1                                  Harvard University                      199
2   Regents of the University of Michigan - Ann Arbor                       28
3                            Arizona State University                       27
4                   University of Colorado at Boulder                       24
5                           Michigan State University                       17
6                     University of Wisconsin-Madison                       16
7                    Florida International University                       16
8                            University of Washington                       16
9                     University of California-Irvine                       15
10                        University of South Florida                       15
11                      University of Texas at Austi

In [50]:
bar_chart_3 = alt.Chart(budget_vs_others).mark_bar().encode(
    x=alt.X('Number of Cancellations:Q', title='Total Budget Cancelled ($)'),
    y=alt.Y('Institution:N', title='Institution',
            sort='-x'
           ),
    color = alt.condition(
        alt.datum.Institution == 'All Others',
        alt.value('lightgray'),
        alt.value('#bf3d2a')
    ),

    tooltip=[
        alt.Tooltip('Institution:N', title='Institution'),
        alt.Tooltip('Total Budget Cancelled:Q', title='Total Budget Cancelled', format='$,.0f')
    ]
).properties(
    title=f'Most {top_n} Affected Institutions by the NSF Grant Cancellations',
    width=600,
    height=400)

bar_chart_3

alt.Chart(...)

Shares the same problem as the one from the last question: the red bars seem too little compared to the 'All the others' one.

#### Most affected institutions with an average line and political affiliations

In [51]:
institution_to_state = df_merged[['org_name', 'org_state']].drop_duplicates()
institution_to_state.columns = ['Institution', 'State']

budget_with_state = pd.merge(budget_by_institution, institution_to_state, on='Institution')

budget_with_state['Party'] = budget_with_state['State'].map(party_map)

top_budget_with_party = budget_with_state.head(15).copy()

In [52]:
bar_chart_political = alt.Chart(top_budget_with_party).mark_bar().encode(
    x=alt.X('Total Budget Cancelled:Q', title='Total Budget Cancelled ($)',
            axis=alt.Axis(format='s')),
    y=alt.Y('Institution:N', sort='-x'),

    color=alt.Color('Party:N',
                    title='Political Party',
                    scale=alt.Scale(domain=['Republican', 'Democrat'],
                                    range=['#bf3d2a', '#4c78a8']),
                    legend=alt.Legend(orient="right")
                   ),

    tooltip=[
        alt.Tooltip('Institution:N'),
        alt.Tooltip('Party:N'),
        alt.Tooltip('Total Budget Cancelled:Q', format='$,.0f')
    ]
).properties(
    title=f'Top {top_n} Most Affected Institutions by Cancelled Budget',
    width=600,
    height=400
)

In [53]:
# Average dotted line and legend
average_line_15 = alt.Chart(budget_by_institution).mark_rule(
    color='black',
    strokeDash=[5, 3],
    size=2
).encode(
    x='average(Total Budget Cancelled):Q'
)

average_value = budget_by_institution['Total Budget Cancelled'].mean()
average_text_formatted = f"${average_value / 1_000_000:.1f}M"

label_text = alt.Chart().mark_text(
    text='Average:',
    align='right',
    fontWeight='bold',
    fontSize=11,
    x=665,
    y=60
)

value_text = alt.Chart().mark_text(
    text=average_text_formatted,
    align='left',
    fontSize=11,
    x=670,
    y=60
)

In [54]:
layered_budget_political = (
    bar_chart_political +
    average_line_15 +
    label_text +
    value_text
)

layered_budget_political

alt.LayerChart(...)

#### Cumulative bar chart

In [55]:
bar_chart_32 = alt.Chart(grouped_budget).mark_bar().encode(
    x=alt.X('Group:N', title='Total Budget Cancelled ($)', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('Value:Q', title='Institution',
            sort='-x'
           ),
    color = alt.Color(
        'Group:N', title='Group',
         scale=alt.Scale(domain=[f'Top {top_n} Institutions', 'All the others'],
         range=['#F2700D', '#d3d3d3']),
        legend=alt.Legend(orient="right")
    ),

    tooltip=[
        alt.Tooltip('Group:N', title='Group'),
        alt.Tooltip('Value:Q', title='Total Budget Cancelled', format='$,.0f')
    ]
).properties(
    width=150,
    height=400)

bar_chart_32

alt.Chart(...)

### Histogram

In [56]:
budget_histogram = alt.Chart(budget_by_institution).mark_bar().encode(
    x=alt.X('Total Budget Cancelled:Q',
            bin=alt.Bin(maxbins=30),
           ),
    y=alt.Y('count()', title='Number of Institutions'),
    color=alt.value('#bf3d2a'),

    tooltip=[
        alt.Tooltip('count()', title='Number of Institutions'),
        alt.Tooltip('Total Budget Cancelled:Q', bin=True, title='Budget Cancellations range')
    ]
).properties(
    title='Distribution of the Budget Cancellations by Institution',
    width=600
)

budget_histogram

alt.Chart(...)

In [57]:
majority_budget = alt.Chart(budget_by_institution).mark_bar().encode(
    x=alt.X('Total Budget Cancelled:Q', bin=alt.Bin(maxbins=30), title=None),
    y=alt.Y('count()', title='Number of Institutions', scale=alt.Scale(domain=[0, 320])),
    color=alt.value('#bf3d2a'),
    tooltip=[
        alt.Tooltip('count()', title='Number of Institutions'),
        alt.Tooltip('Total Budget Cancelled:Q', bin=True, title='Budget Cancellations range')]
).transform_filter(
    alt.datum['Total Budget Cancelled'] < 50000000  # Filtering
).properties(
    title='Distribution of Most Part of the Institutions (< 50.000.000 Worth Value Cancellations)',
    width=350
)

# Outliers' histogram
outliers_budget = alt.Chart(budget_by_institution).mark_bar().encode(
    x=alt.X('Total Budget Cancelled:Q',  bin=alt.Bin(maxbins=12)),
    y=alt.Y('count()', title=None, scale=alt.Scale(domain=[0, 1.2])),
    color=alt.value('#bf3d2a'),
    tooltip=[
        alt.Tooltip('count()', title='Number of Institutions'),
        alt.Tooltip('Total Budget Cancelled:Q', bin=True, title='Cancellations range')]
).transform_filter(
    alt.datum['Total Budget Cancelled'] >= 50000000
).properties(
    title='Outliers (≥ 50.000.000 Worth Value Cancellations)',
    width=300
)

combined_budget = majority_budget | outliers_budget

combined_budget

alt.HConcatChart(...)

We can quote the same problem as the last questions' histogram: the main problem lays in the different values of the Y axis leading to a possible confusion by the user.

### Jitter plot

In [58]:
jitter_plot_econ = alt.Chart(budget_by_institution).mark_circle(
    opacity=0.6,
    size=80
).encode(
    x=alt.X('Total Budget Cancelled:Q'),

    y=alt.Y('jitter:Q',
            title=None,
            axis=alt.Axis(values=[0], ticks=False, grid=False, labels=False)
           ),
    color=alt.value('#bf3d2a'),
    tooltip=[
        alt.Tooltip('Institution:N'),
        alt.Tooltip('Total Budget Cancelled:Q')
    ]
).transform_calculate(
    jitter='random()'
).properties(
    title='Distribution of Economic Loss due the NSF Grant Cancellations (Jitter Plot)',
    width=850,
    height=150
)

average_line_strip_econ = alt.Chart(budget_by_institution).mark_rule(
    color='lightgrey',
    strokeDash=[5, 3],
    size=2
).encode(
    x='average(Total Budget Cancelled):Q'
)

layered_strip_plot_econ = (jitter_plot_econ + average_line_strip_econ)

layered_strip_plot_econ

alt.LayerChart(...)

### Pyramid chart

#### Basic Pyramid Chart

In [59]:
comparison_df = pd.merge(cancellations_by_institution, budget_by_institution, on='Institution')

top_15_count = cancellations_by_institution.head(15)['Institution'].tolist()
top_15_budget = budget_by_institution.head(15)['Institution'].tolist()

institutions_to_show = list(set(top_15_count + top_15_budget))
final_comparison_data = comparison_df[comparison_df['Institution'].isin(institutions_to_show)].copy()

In [60]:
data_for_pyramid = final_comparison_data.melt(
    id_vars=['Institution'],
    value_vars=['Number of Cancellations', 'Total Budget Cancelled'],
    var_name='Metric',
    value_name='Value'
)

sort_order = final_comparison_data.sort_values('Total Budget Cancelled', ascending=False)['Institution'].tolist()

middle = alt.Chart(final_comparison_data).mark_text(align='center').encode(
    y=alt.Y('Institution:N', axis=None, sort=sort_order),
    text='Institution:N'
).properties(
    width=250
)

left = alt.Chart(data_for_pyramid).mark_bar().encode(
    y=alt.Y('Institution:N', axis=None, sort=sort_order),
    x=alt.X('Value:Q', title='Number of Cancellations', sort='descending'),
    color=alt.value('#bf3d2a'),
    tooltip=[
        alt.Tooltip('Institution:N'),
        alt.Tooltip('Value:Q', title='Number of Cancellations', format=',')
    ]
).transform_filter(
    alt.datum.Metric == 'Number of Cancellations'
).properties(
    title='By Grant Cancellations Number'
)

right = alt.Chart(data_for_pyramid).mark_bar().encode(
    y=alt.Y('Institution:N', axis=None, sort=sort_order),
    x=alt.X('Value:Q', title='Total Budget Cancelled ($)', axis=alt.Axis(format='s')),
    color=alt.value('#4c78a8'),
    tooltip=[
        alt.Tooltip('Institution:N'),
        alt.Tooltip('Value:Q', title='Total Budget Cancelled', format='$,.0f')
    ]
).transform_filter(
    alt.datum.Metric == 'Total Budget Cancelled'
).properties(
    title='By Economic Impact'
)

In [61]:
pyramid_chart = alt.concat(
    left,
    middle,
    right,
    spacing=10
).properties(
    title='Comparison of the Most Affected Institutions: Cancelled Grants vs. Economic Loss'
).resolve_scale(
    x='independent'
).configure_title(anchor = 'middle')

pyramid_chart

alt.ConcatChart(...)

Maybe we can change the color hue based on the states' political parties because there might be confusions.   

#### Political Party Pyramid Chart

In [62]:
institution_to_state = df_merged[['org_name', 'org_state']].drop_duplicates()
institution_to_state.columns = ['Institution', 'State']

final_comparison_with_party = pd.merge(final_comparison_data, institution_to_state, on='Institution')

final_comparison_with_party['Party'] = final_comparison_with_party['State'].map(party_map)

In [63]:
sort_order = final_comparison_with_party.sort_values('Total Budget Cancelled', ascending=False)['Institution'].tolist()

middle = alt.Chart(final_comparison_with_party).mark_text(align='center').encode(
    y=alt.Y('Institution:N', axis=None, sort=sort_order),
    text='Institution:N'
).properties(
    width=250
)

In [64]:
left_bars = alt.Chart(final_comparison_with_party).mark_bar().encode(
    y=alt.Y('Institution:N', axis=None, sort=sort_order),
    x=alt.X('Number of Cancellations:Q', title='Number of Cancellations', sort='descending'),

    color=alt.Color('Party:N', legend=None,
                    scale=alt.Scale(domain=['Republican', 'Democrat'],
                                    range=['#bf3d2a', '#4c78a8'])),

    tooltip=[alt.Tooltip('Institution:N'), alt.Tooltip('Party:N'), alt.Tooltip('Number of Cancellations:Q', format=',')]
).properties(
    title='By Number of Grants'
)

left_average_line = alt.Chart(comparison_df).mark_rule(color='black', strokeDash=[3,3], size=2).encode(
    x='average(Number of Cancellations):Q'
)

label_text_left = alt.Chart().mark_text(
    text='Average:',
    align='right',
    fontWeight='bold',
    fontSize=11,
    dx=-95,
    dy=180
)

left_average_text = alt.Chart(comparison_df).mark_text(
    align='right',
    dx=-225,
    dy=180,
    color='black',
    fontSize=11
).encode(
    x='average(Number of Cancellations):Q',
    text=alt.Text('average(Number of Cancellations):Q', format='.1f')
)

left = left_bars + left_average_line + label_text_left + left_average_text

In [65]:
right_bars = alt.Chart(final_comparison_with_party).mark_bar().encode(
    y=alt.Y('Institution:N', axis=None, sort=sort_order),
    x=alt.X('Total Budget Cancelled:Q', title='Total Budget Cancelled ($)', axis=alt.Axis(format='s')),

    color=alt.Color('Party:N', legend=None,
                    scale=alt.Scale(domain=['Republican', 'Democrat'],
                                    range=['#bf3d2a', '#4c78a8'])),

    tooltip=[alt.Tooltip('Institution:N'), alt.Tooltip('Party:N'), alt.Tooltip('Total Budget Cancelled:Q', format='$,.0f')]
).properties(
    title='By Economic Impact'
)

right_average_line = alt.Chart(comparison_df).mark_rule(color='black', strokeDash=[3,3], size=2).encode(
    x='average(Total Budget Cancelled):Q'
)

label_text_right = alt.Chart().mark_text(
    text='Average:',
    align='left',
    fontWeight='bold',
    fontSize=11,
    dx=67,
    dy=180
)

right_average_text = alt.Chart(comparison_df).mark_text(
    align='left',
    dx=260,
    dy=180,
    color='black',
    fontSize=11
).encode(
    x='average(Total Budget Cancelled):Q',
    text=alt.Text('average(Total Budget Cancelled):Q', format='$.2s')
)

right = right_bars + right_average_line + label_text_right + right_average_text

In [66]:
pyramid_chart_political = alt.concat(
    left,
    middle,
    right,
    spacing=10
).properties(
    title='Comparison Between Cancelled Grants vs. Budget by Institutions and Political Affiliation'
).resolve_scale(
    x='independent'
)

pyramid_chart_political.configure_title(anchor = 'middle')

alt.ConcatChart(...)

This pyramid chart where we combine the cancelled grants and budgets seems perfect to save space and, at the same time, see the correlation between the number of cancellations and its monetary value. It will be great for the general visualization.

### Final Visualization

In [67]:
layered_budget_political

alt.LayerChart(...)

#### Justification
As in the previous question, a **horizontal bar chart** is again employed to answer Q3, but with a shift in the primary metric. This chart moves the analysis from the volume of cancellations to their **economic impact**, using the total budget cancelled of each institution as the main variable. This approach is essential for understanding the financial magnitude of the losses.

To facilitate the analysis, the chart incorporates the same two contextual layers as before. Color hue encodes the political affiliation of the institution's state, enabling a direct comparison with the Q2 chart to see if the political pattern changes when focusing on budget instead of grant count. Furthermore, a dotted line and an explicit annotation denote the average cancelled budget across all institutions. This benchmark is essential for contextualizing the scale of the financial losses, specialy for acknowledging the impact to these top-tier institutions

This view allows to identify which institutions suffered the **greatest financial cancellations** and observe the political distribution of this economic impact. For example, a user can immediately see the huge financial loss at institutions like UCLA and Harvard compared to the overall institutional average of $3.4M.

## Q4: Is there any correlation between the cancelled grants and the list of flagged words?

### Bar charts

In [68]:
# Load the flagged words by Trump administration list
flagged_words_df = pd.read_csv('flagged_words_trump_admin.csv')
flagged_words_set = set(flagged_words_df['flagged_word'])

print(f"There is {len(flagged_words_set)} flagged words. ")

There is 54 flagged words. 


In [69]:
df_merged['abstract_clean'] = df_merged['abstract'].fillna('').str.lower()

def contains_flagged_word(text, flagged_words):
    """
    We search for a flagged word in the abstracts.
    """
    for word in flagged_words:
        if word in text:
            return True
    return False

df_merged['has_flagged_word'] = df_merged['abstract_clean'].apply(
    lambda abstract: contains_flagged_word(abstract, flagged_words_set)
)

print("Final counter of cancelled grant with and without 'flagged words':")
print(df_merged['has_flagged_word'].value_counts())

Final counter of cancelled grant with and without 'flagged words':
has_flagged_word
True     1787
False     183
Name: count, dtype: int64


In [70]:
general_summary = df_merged['has_flagged_word'].value_counts().reset_index()
general_summary.columns = ['Contains Flagged Word', 'Number of Grants']

general_summary['Contains Flagged Word'] = general_summary['Contains Flagged Word'].map({True: 'Yes', False: 'No'})

# Percentage of abstracts with flagged words
percentage_flaggedw = (1787 * 100) / 1970
percentage_text = f"{percentage_flaggedw:.1f}"

label_flagged = alt.Chart().mark_text(
    text=f'{percentage_text}%',
    align='right',
    fontWeight='bold',
    color = 'white',
    fontSize=20,
    x=80,
    y=70
)

# Percentage of abstracts without flagged words
percentage_text2 = f"{100 - percentage_flaggedw:.1f}"

label_not_flagged = alt.Chart().mark_text(
    text=f'{percentage_text2}%',
    align='right',
    fontWeight='bold',
    color = '#4f4f4f',
    fontSize=15,
    x=70,
    y=25
)

# The proportion of cancelled grants that included flagged words in their abstracts
stacked_bar_chart = alt.Chart(general_summary).mark_bar().encode(
    y=alt.Y('Number of Grants:Q'),

    color=alt.Color('Contains Flagged Word:N',
                    title=['Did the abstract', 'contain a flagged', 'word?'],
                    scale=alt.Scale(domain=['No', 'Yes'], range=['#d3d3d3', '#965493']),
                    legend=alt.Legend(orient="right")
                   ),

    tooltip=['Contains Flagged Word:N', 'Number of Grants:Q']
).properties(
    title=['Proportion of Cancelled Grants', 'with and without Flagged Words'],
    height=400,
    width = 100
)

proportion_chart = stacked_bar_chart + label_flagged + label_not_flagged
proportion_chart

alt.LayerChart(...)

#### Most common flagged words

In [71]:
df_merged['abstract_clean'] = df_merged['abstract'].fillna('').str.lower()

def find_all_flagged_words(text, flagged_words):
    found_words = []
    for word in flagged_words:
        if word in text:
            found_words.append(word)
    return found_words

df_merged['found_words'] = df_merged['abstract_clean'].apply(
    lambda abstract: find_all_flagged_words(abstract, flagged_words_set)
)

exploded_words = df_merged.explode('found_words')

# We count the most frequent flagged words
top_words = exploded_words['found_words'].value_counts().nlargest(20).reset_index()
top_words.columns = ['Flagged Word', 'Frequency']

# Bar chart
detail_chart = alt.Chart(top_words).mark_bar(color='#bf3d2a').encode(
    x=alt.X('Frequency:Q', title='Frequency of Ocurrence'),
    y=alt.Y('Flagged Word:N', title='Key word', sort='-x'),
    tooltip=['Flagged Word:N', 'Frequency:Q']
).properties(
    title='Top 20 Most Frequent Key Words in Cancelled Grants',
    width=600,
    height=400
)

detail_chart

alt.Chart(...)

#### Counter of Flag Words per Grant

In [72]:
def count_all_flagged_words(text, flagged_words):
  """
  Find all the flagged words in the abstract and count them.
  """
  total_count = 0
  for word in flagged_words:
      matches = re.findall(r'\b' + re.escape(word) + r'\b', text)
      total_count += len(matches)
  return total_count

df_merged['flagged_word_count'] = df_merged['abstract_clean'].apply(
    lambda abstract: count_all_flagged_words(abstract, flagged_words_set)
)

print(df_merged.sort_values(by='flagged_word_count', ascending=False)[['project_title', 'flagged_word_count']].head())

word_count_by_institution = df_merged.groupby('org_name')['flagged_word_count'].sum()

word_count_by_institution = word_count_by_institution.sort_values(ascending=False).reset_index()
word_count_by_institution.columns = ['Institution', 'Total Flagged Words']

top_n = 15
top_institutions_by_words = word_count_by_institution.head(top_n)

print(top_institutions_by_words)

                                         project_title  flagged_word_count
847  ADVANCE Adaptation: Project SAGES: Striving to...                  36
66   ADVANCE Partnership: Advancing Gender Equity i...                  36
857  ADVANCE Adaptation: CSU STEPS for Gender Equit...                  35
664  Collaborative Research: Recognition of Gender ...                  34
55   ADVANCE Adaptation: Creating a Destination of ...                  34
                                          Institution  Total Flagged Words
0                University of California-Los Angeles                  471
1                                  Harvard University                  334
2                   University of Colorado at Boulder                  246
3   Regents of the University of Michigan - Ann Arbor                  230
4                            Arizona State University                  209
5                    Florida International University                  186
6                        

In [73]:
word_count_chart = alt.Chart(top_institutions_by_words).mark_bar().encode(
    x=alt.X('Total Flagged Words:Q', title='Total Number of Mentioned Flagged Words'),

    y=alt.Y('Institution:N',
            title='Institution',
            sort='-x'
           ),

    tooltip=[
        alt.Tooltip('Institution:N'),
        alt.Tooltip('Total Flagged Words:Q')
    ]
).properties(
    title='The Institutions with the Most Flagged Words in their Abstracts',
    width=600,
    height=400
)

word_count_chart

alt.Chart(...)

This bar chart shows the institutions that contain in their abstracts the most flagged words. It could help to study the correlation between the words and the cancellations, specially if an institution ranks high in both charts.

#### Density of Words per Grant

In [74]:
institutions_to_show = [
    'University of California-Los Angeles',
    'Harvard University',
    'University of Colorado at Boulder',
    'Regents of the University of Michigan - Ann Arbor',
    'Arizona State University',
    'Florida International University',
    'Colorado State University',
    'University of Texas at Austin',
    'University of Washington',
    'University of Wisconsin-Madison',
    'University of Illinois at Urbana-Champaign',
    'Michigan State University',
    'University of South Florida',
    'University of Texas at El Paso',
    'George Mason University'
]

In [75]:
density_df = df_merged.groupby('org_name').agg(
    total_words=('flagged_word_count', 'sum'),
    grant_count=('grant_id', 'size')
).reset_index()

density_df.columns = ['Institution', 'total_words', 'grant_count']
density_df['Words per Grant'] = density_df['total_words'] / density_df['grant_count']

density_df = density_df[density_df['grant_count'] > 5]

filtered_density_df = density_df[density_df['Institution'].isin(institutions_to_show)]

words_density = alt.Chart(filtered_density_df).mark_bar().encode(
    x=alt.X('Words per Grant:Q', title='Words per Grant'),

    y=alt.Y('Institution:N',
            title='Institution',
            sort='-x'
           ),

    tooltip=[
        alt.Tooltip('Institution:N'),
        alt.Tooltip('Words per Grant:Q', title='Words per Grant', format='.2f'),
        alt.Tooltip('total_words:Q', title='Total Flagged Words'),
        alt.Tooltip('grant_count:Q', title='Grant Count')
    ]
).properties(
    title='Institutions by Words per Grant',
    width=600,
    height=400
)

words_density

alt.Chart(...)

This chart is also important as it shows the frequency that the flagged words appeared in the abstracts.

### Scatter plot

In [77]:
combined_data = pd.merge(top_institutions_by_words, filtered_density_df, on='Institution')

scatterplot_comparison = alt.Chart(combined_data).mark_point(
    filled=True,
    size=200,
    opacity=0.7
).encode(
    x=alt.X('Total Flagged Words:Q', title='Total Number of Mentioned Flagged Words'),
    y=alt.Y('Words per Grant:Q', title='Average Mentions of Flagged Words per Institution'),

    tooltip=[
        alt.Tooltip('Institution:N'),
        alt.Tooltip('Total Flagged Words:Q', format=','),
        alt.Tooltip('Words per Grant:Q', title='Average per Grant', format='.2f')
    ]
).properties(
    title='Volume vs. Density of Flagged Words per Institution',
    width=600,
    height=400
).interactive()

scatterplot_comparison

alt.Chart(...)

It's interesting what this scatterplot shows. A high average of flagged words in an institution doesn't mean these appear very often in their abstracts. We can see the example of the University of California as they are the institution with the most average of flagged words but also the one with the least density.  

### Slope chart

In [78]:
combined_data['Rank_Total'] = combined_data['Total Flagged Words'].rank(ascending=False)
combined_data['Rank_Density'] = combined_data['Words per Grant'].rank(ascending=False)

data_for_slope = combined_data.melt(
    id_vars='Institution',
    value_vars=['Rank_Total', 'Rank_Density'],
    var_name='Metric',
    value_name='Rank'
)

slope_lines = alt.Chart(data_for_slope).mark_line().encode(
    x='Metric:N',
    y='Rank:Q',
    color='Institution:N',
    detail='Institution:N'
)

slope_points = alt.Chart(data_for_slope).mark_circle(size=100).encode(
    x='Metric:N',
    y='Rank:Q',
    color='Institution:N',
    tooltip=['Institution:N', 'Rank:Q']
)

slope_chart = (slope_lines + slope_points).encode(
    y=alt.Y('Rank:Q', scale=alt.Scale(reverse=True))
).properties(
    title='Ranking\'s Variation: Volume vs. Density of Flagged Words',
    width=300
).interactive()

slope_chart

alt.LayerChart(...)

It's very interessting to see the ranking's changes between the volume and density but it's not easy to follow the lines having 15 institutions (and the colors seem to repeat or to be very similar).

### Pyramid chart

In [79]:
combined_data_q4 = pd.merge(top_institutions_by_words, filtered_density_df, on='Institution')

data_for_pyramid_2 = combined_data_q4.melt(
    id_vars=['Institution'],
    value_vars=['Total Flagged Words', 'Words per Grant'],
    var_name='Metric',
    value_name='Value'
)

sort_order = top_institutions_by_words['Institution'].tolist()
print(data_for_pyramid_2.head())

                                         Institution               Metric  \
0               University of California-Los Angeles  Total Flagged Words   
1                                 Harvard University  Total Flagged Words   
2                  University of Colorado at Boulder  Total Flagged Words   
3  Regents of the University of Michigan - Ann Arbor  Total Flagged Words   
4                           Arizona State University  Total Flagged Words   

   Value  
0  471.0  
1  334.0  
2  246.0  
3  230.0  
4  209.0  


In [80]:
middle_2 = alt.Chart(data_for_pyramid_2).mark_text(align='center').encode(
    y=alt.Y('Institution:N', axis=None, sort=sort_order),
    text='Institution:N'
).properties(
    width=250,
    height = 400
)


left_2 = alt.Chart(data_for_pyramid_2).mark_bar().encode(
    y=alt.Y('Institution:N', axis=None, sort=sort_order),
    x=alt.X('Value:Q',
            title='Total Number of Flagged Words',
            sort='descending'
           ),

    color=alt.value('#bf3d2a'),

    tooltip=[
        alt.Tooltip('Institution:N'),
        alt.Tooltip('Value:Q', title='Total Flagged Words', format=',')
    ]
).transform_filter(
    alt.datum.Metric == 'Total Flagged Words'
).properties(
    title='Total Number',
    height = 400
)


right_2 = alt.Chart(data_for_pyramid_2).mark_bar().encode(
    y=alt.Y('Institution:N', axis=None, sort=sort_order),
    x=alt.X('Value:Q', title='Average of Flagged Words per Institution\'s Grants'),

    color=alt.value('#4c78a8'),

    tooltip=[
        alt.Tooltip('Institution:N'),
        alt.Tooltip('Value:Q', title='Average per Institution', format='.2f')
    ]
).transform_filter(
    alt.datum.Metric == 'Words per Grant'
).properties(
    title='Average Density',
    height = 400
)

In [81]:
pyramid_chart_2 = alt.concat(
    left_2,
    middle_2,
    right_2,
    spacing=5
).properties(
    title='Comparison of Total number of Flagged Words vs. its Density by Institution'
)

pyramid_chart_2

alt.ConcatChart(...)

It ends up being very useful at the time of camparison to but a bar chart next to the other in a pyramid chart form. This way we can see more clearly both rankings.

### Final visualization

In [82]:
proportion_chart

alt.LayerChart(...)

#### Justification
To answer Q4, a **stacked bar chart** was selected as the most direct and effective visualization. This chart type is optimal for saving space and **combine the categorical information**.

The single bar represents the entirety of the cancelled grants, segmented by the presence or absence of flagged terms. To ensure clarity and immediate comprehension, each segment is explicitly annotated with its percentage value. This eliminates potential ambiguities and allows the user to immediately understand the magnitude of each section.

This visualization provides clear answers: a user can see the overwhelming dominance of the 'Yes' category (90.7%), indicating a **strong apparent correlation** between the use of these terms and a grant's cancelled status. While this chart answers the actual question, complementary analyses of word frequency, institutional density (explored previously) or even data about the non-cancelled grants (which we do not have) seem crucial for providing **more context** to the topic.

## Q5: Is there any correlation between the cancelled grants and the list of grants in Cruz's list? And with respect to reinstated grants?

### Bar chart

In [83]:
summary_cruz = df_merged.groupby('in_cruz_list').agg(
    grant_count=('grant_id', 'size'), # Number of grants in each group
    total_budget=('nsf_total_budget', 'sum') # Sum of the budget of each group
).reset_index()

summary_cruz['In Cruz List'] = summary_cruz['in_cruz_list'].map({
    True: 'In Cruz\'s list',
    False: 'Not in Cruz\'s list'
})

total_budget_cancelled = summary_cruz['total_budget'].sum()
summary_cruz['Percentage'] = (summary_cruz['total_budget'] / total_budget_cancelled) * 100
summary_cruz['TextLabel'] = summary_cruz['Percentage'].apply(lambda x: f'{x:.1f}%')

print(summary_cruz)

  in_cruz_list  grant_count  total_budget        In Cruz List  Percentage  \
0        False         1503    1394118667  Not in Cruz's list   81.191381   
1         True          467     322958508      In Cruz's list   18.808619   

  TextLabel  
0     81.2%  
1     18.8%  


/tmp/ipython-input-1366860177.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary_cruz = df_merged.groupby('in_cruz_list').agg(


In [84]:
bar_chart_count = alt.Chart(summary_cruz).mark_bar().encode(
    x=alt.X('grant_count:Q', title='Number of Cancelled Grants'),
    y=alt.Y('In Cruz List:N', title=None, sort='-x'),
    color=alt.Color('In Cruz List:N',
                    legend=None,
                    scale=alt.Scale(range=['#e28779', '#862b1d'])
                   ),
    tooltip=[
        alt.Tooltip('In Cruz List:N', title='Grupo'),
        alt.Tooltip('grant_count:Q', title='Number of Grants', format=',')
    ]
).properties(
    title='Number of Cancelled Grants'
)

bar_chart_count

alt.Chart(...)

In [85]:
reinstated_cruz_summary = df_merged.groupby(['reinstated', 'in_cruz_list']).size().reset_index(name='count')

reinstated_cruz_summary['reinstated'] = reinstated_cruz_summary['reinstated'].map({True: 'Reinstated', False: 'Don\'t reinstated'})
reinstated_cruz_summary['in_cruz_list'] = reinstated_cruz_summary['in_cruz_list'].map({True: 'In Cruz\'s list', False: 'Not in Cruz\'s list'})

group_totals = reinstated_cruz_summary.groupby('reinstated')['count'].transform('sum')
reinstated_cruz_summary['Percentage based on reintegration'] = (reinstated_cruz_summary['count'] / group_totals) * 100
reinstated_cruz_summary['Percentage based on reintegration'] = reinstated_cruz_summary['Percentage based on reintegration'].round(2)

reinstated_cruz_summary['TextLabel'] = reinstated_cruz_summary['Percentage based on reintegration'].apply(lambda x: f'{x:.1f}%')

print(reinstated_cruz_summary)

         reinstated        in_cruz_list  count  \
0  Don't reinstated  Not in Cruz's list   1122   
1  Don't reinstated      In Cruz's list    428   
2        Reinstated  Not in Cruz's list    381   
3        Reinstated      In Cruz's list     39   

   Percentage based on reintegration TextLabel  
0                              72.39     72.4%  
1                              27.61     27.6%  
2                              90.71     90.7%  
3                               9.29      9.3%  


/tmp/ipython-input-1175512505.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  reinstated_cruz_summary = df_merged.groupby(['reinstated', 'in_cruz_list']).size().reset_index(name='count')


In [86]:
reinstated_chart = alt.Chart(reinstated_cruz_summary).mark_bar().encode(
    x=alt.X('reinstated:N', title='Status', axis=alt.Axis(labelAngle=0)),
        y=alt.Y('count:Q', title='Number of Grants'),
        color=alt.Color('in_cruz_list:N', title='In Cruz\'s list?', scale=alt.Scale(range=['#e28779', '#862b1d'])),
        xOffset='in_cruz_list:N',

    tooltip=['reinstated:N', 'in_cruz_list:N', 'count:Q', 'Percentage based on reintegration:Q']
).properties(
    title="Correlation Between Reinstated Grants and Cruz List",
    width=300
)

reinstated_chart

alt.Chart(...)

This chart is ideal to grasp the real magnitude of each group as we can see clearly that the combination of "Not in Cruz's list" and "Don't reinstated" is the highest but, at the same time is not a good option for comparing proportions.

In [87]:
reinstated_chart_normalized = alt.Chart(reinstated_cruz_summary).mark_bar().encode(
    x=alt.X('reinstated:N', title='Status', axis=alt.Axis(labelAngle=0)),

    y=alt.Y('count:Q',
            title='Proportion of Grants',
            stack='normalize',
            axis=alt.Axis(format='%')
           ),
    color=alt.Color('in_cruz_list:N', title='In Cruz\'s list?', scale=alt.Scale(range=['#e28779', '#862b1d'])),

    tooltip=['reinstated:N', 'in_cruz_list:N', 'count:Q', 'Percentage based on reintegration:Q']
).properties(
    title="Proportional Correlation Between Reinstated Grants and Cruz List",
    width=300
)

reinstated_chart_normalized

alt.Chart(...)

Good for proportions but hides the real values of the categories.

### Heatmap

In [88]:
heatmap = alt.Chart(reinstated_cruz_summary).mark_rect().encode(
    x=alt.X('reinstated:N', title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y('in_cruz_list:N', title=None),
    color=alt.Color('count:Q',
                    title='Number of Grants',
                    scale=alt.Scale(scheme='goldorange')
                   ),
    tooltip=['reinstated:N', 'in_cruz_list:N', 'count:Q', 'TextLabel:N']
)

text = alt.Chart(reinstated_cruz_summary).mark_text(
    color='white',
    fontSize=17,
    fontWeight='bold'
).encode(
    x='reinstated:N',
    y='in_cruz_list:N',
    text=alt.Text('count_and_percent:N')
).transform_calculate(
    count_and_percent="[datum.count, datum.TextLabel]"
)

q5_heatmap = (heatmap + text).properties(
    title='Crossed Analysis: Reinstatements vs. Cruz\'s list',
    width=500,
    height=400
)

q5_heatmap

alt.LayerChart(...)

### Final Visualization

In [89]:
q5_heatmap

alt.LayerChart(...)

#### Justification
For the last question, an **annotated heatmap** was designed. This 2x2 matrix visualization is the optimal choice as it is capable of **displaying the relationship** between the two categorical variables **without sacrificing critical information**.

The design has a **good data-ink ratio**. Each of the four cells represents one of the combinations of reinstatement status and appearance on the Cruz's list. The absolute number of grants in each square is encoded dually for emphasis: through color intensity and as an explicit numerical label. Additionally, to provide a proportional insight, a second text annotation displays the conditional percentage within each status column.

Alternatives were deemed inferior: a grouped bar chart obscures the vital proportional comparison, while a stacked bar chart completely hides the absolute counts and the scale of the issue.

With this heatmap, a user can instantly make two key observations: by comparing percentages horizontally, they can see that **a grant was far more likely to be reinstated if it was not on Cruz's list** (90.7%); simultaneously, by observing the absolute counts, it's visible that the total number of reinstated grants corresponds to a small fraction of those that were not, providing interesting context.

# General Visualization

In [90]:
dichromatic_map & pyramid_chart_political & (proportion_chart | q5_heatmap)

alt.VConcatChart(...)

###Justification

For the final visualization, we combined all the previously generated charts into a single layout. This allows the reader to understand all the key insights at a glance. It is also convenient that **the analyses for Questions 2 and 3 can be addressed simultaneously with one combined chart**. Meanwhile, the charts for Questions 4 and 5 are smaller and more simple, which makes them well-suited to being displayed side by side without losing clarity.